# Representing Text as Tensors

**AI3 · NLP Deep Dive** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~15 min

## Text Classification

Throughout the first part of this section, we will focus on **text classification** task. We will use the [AG News](https://www.kaggle.com/amananandrai/ag-news-classification-dataset) Dataset, which contains news articles like the following:

* Category: Sci/Tech
* Title: Ky. Company Wins Grant to Study Peptides (AP)
* Body: AP - A company founded by a chemistry researcher at the University of Louisville won a grant to develop...

Our goal will be to classify the news item into one of the categories based on text.



## Representing text

If we want to solve Natural Language Processing (NLP) tasks with neural networks, we need some way to represent text as tensors. Computers already represent textual characters as numbers that map to fonts on your screen using encodings such as ASCII or UTF-8.

![Image showing diagram mapping a character to an ASCII and binary representation](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/5-NLP/13-TextRep/images/ascii-character-map.png)

> [Image source](https://www.seobility.net/en/wiki/ASCII)

As humans, we understand what each letter **represents**, and how all characters come together to form the words of a sentence. However, computers by themselves do not have such an understanding, and neural network has to learn the meaning during training.

Therefore, we can use different approaches when representing text:

* **Character-level representation**, when we represent text by treating each character as a number. Given that we have *C* different characters in our text corpus, the word *Hello* would be represented by 5x*C* tensor. Each letter would correspond to a tensor column in one-hot encoding.
* **Word-level representation**, in which we create a **vocabulary** of all words in our text, and then represent words using one-hot encoding. This approach is somehow better, because each letter by itself does not have much meaning, and thus by using higher-level semantic concepts - words - we simplify the task for the neural network. However, given the large dictionary size, we need to deal with high-dimensional sparse tensors.

Regardless of the representation, we first need to convert the text into a sequence of **tokens**, one token being either a character, a word, or sometimes even part of a word. Then, we convert the token into a number, typically using **vocabulary**, and this number can be fed into a neural network using one-hot encoding.



## N-Grams

In natural language, precise meaning of words can only be determined in context. For example, meanings of *neural network* and *fishing network* are completely different. One of the ways to take this into account is to build our model on pairs of words, and considering word pairs as separate vocabulary tokens. In this way, the sentence *I like to go fishing* will be represented by the following sequence of tokens: *I like*, *like to*, *to go*, *go fishing*. The problem with this approach is that the dictionary size grows significantly, and combinations like *go fishing* and *go shopping* are presented by different tokens, which do not share any semantic similarity despite the same verb.  

In some cases, we may consider using tri-grams -- combinations of three words -- as well. Thus the approach is such is often called **n-grams**. Also, it makes sense to use n-grams with character-level representation, in which case n-grams will roughly correspond to different syllabi.



## Bag-of-Words and TF/IDF

When solving tasks like text classification, we need to be able to represent text by one fixed-size vector, which we will use as an input to final dense classifier. One of the simplest ways to do that is to combine all individual word representations, eg. by adding them. If we add one-hot encodings of each word, we will end up with a vector of frequencies, showing how many times each word appears inside the text. Such representation of text is called **bag of words** (BoW).

![Illustration](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/5-NLP/13-TextRep/images/bow.png)

> Image by the author

A BoW essentially represents which words appear in text and in which quantities, which can indeed be a good indication of what the text is about. For example, news article on politics is likely to contains words such as *president* and *country*, while scientific publication would have something like *collider*, *discovered*, etc. Thus, word frequencies can in many cases be a good indicator of text content.

The problem with BoW is that certain common words, such as *and*, *is*, etc. appear in most of the texts, and they have highest frequencies, masking out the words that are really important. We may lower the importance of those words by taking into account the frequency at which words occur in the whole document collection. This is the main idea behind TF/IDF approach, which is covered in more detail in the notebooks attached to this lesson.

However, none of those approaches can fully take into account the **semantics** of text. We need more powerful neural networks models to do this, which we will discuss later in this section.



## Exercises: Text Representation

Continue your learning in the following notebooks:

* [Text Representation with PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/13-TextRep/TextRepresentationPyTorch.ipynb)
* [Text Representation with TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/13-TextRep/TextRepresentationTF.ipynb)



## Conclusion

So far, we have studied techniques that can add frequency weight to different words. They are, however, unable to represent meaning or order. As the famous linguist J. R. Firth said in 1935, "The complete meaning of a word is always contextual, and no study of meaning apart from context can be taken seriously." We will learn later in the course how to capture contextual information from text using language modeling.



## Challenge

Try some other exercises using bag-of-words and different data models. You might be inspired by this [competition on Kaggle](https://www.kaggle.com/competitions/word2vec-nlp-tutorial/overview/part-1-for-beginners-bag-of-words)



## Review & Self Study

Practice your skills with text embeddings and bag-of-words techniques on [Microsoft Learn](https://learn.microsoft.com/learn/modules/intro-natural-language-processing-pytorch/?WT.mc_id=academic-77998-cacaste)



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — L2-normalize embedding rows

●○○ · ~8 min

> Cosine similarity, retrieval and clustering on embeddings usually start by L2-normalizing rows, so that the dot product equals the cosine. Edge case: a zero vector has norm 0, so dividing by it gives NaN; leave such rows as zeros.

Implement `l2_normalize(X)`: return a float array where every row has unit L2 norm. Rows that are all zeros must stay all zeros.

<details><summary>Hint</summary>

n = np.linalg.norm(X, axis=1, keepdims=True); divide by np.where(n == 0, 1.0, n) and mask the zero rows back to 0.

</details>


In [ ]:
import numpy as np

def l2_normalize(X):
    X = np.asarray(X, dtype=float)
    # your code
    return X

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    X = rng.normal(0, 1, (int(rng.integers(3, 12)), int(rng.integers(2, 8))))\n    if rng.random() < 0.4:\n        X[int(rng.integers(0, X.shape[0]))] = 0.0   # a dead embedding\n    return {\"X\": X}\n\ndef l2_normalize(X):\n    X = np.asarray(X, dtype=float)\n    n = np.linalg.norm(X, axis=1, keepdims=True)\n    return np.where(n == 0, 0.0, X / np.where(n == 0, 1.0, n))", "l2_normalize", 20)

### Task 2 — Pairwise cosine similarity matrix

●○○ · ~10 min

> Given normalized embeddings, cosine similarity is a single matrix multiply: `An @ Bn.T` gives all pairwise scores in one shot. This is the core of retrieval, deduplication and clustering baselines. Guard zero vectors so you never emit NaN.

Implement `cosine_sim(A, B)` → a (len(A), len(B)) float matrix of cosine similarities between every row of `A` and every row of `B`. If either vector is all zeros, the similarity is 0.

<details><summary>Hint</summary>

Normalize the rows of both matrices with the guarded-division trick, then multiply An @ Bn.T.

</details>


In [ ]:
import numpy as np

def cosine_sim(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    # your code — normalise each row first
    return A @ B.T  # wrong — dot products, not cosine

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n, m, d = int(rng.integers(2, 9)), int(rng.integers(2, 9)), int(rng.integers(2, 7))\n    A = rng.normal(0, 1, (n, d)); B = rng.normal(0, 1, (m, d))\n    if rng.random() < 0.3:\n        A[int(rng.integers(0, n))] = 0.0\n    if rng.random() < 0.3:\n        B[int(rng.integers(0, m))] = 0.0   # zero-norm rows on BOTH sides\n    return {\"A\": A, \"B\": B}\n\ndef cosine_sim(A, B):\n    A = np.asarray(A, dtype=float); B = np.asarray(B, dtype=float)\n    an = np.linalg.norm(A, axis=1, keepdims=True)\n    bn = np.linalg.norm(B, axis=1, keepdims=True)\n    An = np.where(an == 0, 0.0, A / np.where(an == 0, 1.0, an))\n    Bn = np.where(bn == 0, 0.0, B / np.where(bn == 0, 1.0, bn))\n    return An @ Bn.T", "cosine_sim", 20)

### Task 3 — Regex-free tokenizer

●○○ · ~8 min

> Half of NLP olympiad bugs are tokenization bugs. The regex-free baseline: lowercase the text, keep maximal runs of alphanumeric characters, and treat every other character as a separator. Simple, predictable, and fully under your control.

Implement `tokenize(text)` → list of tokens. Lowercase `text`, keep runs of letters/digits (`str.isalnum`), split on every other character. Example: `'Cat,DOG-9!'` → `['cat', 'dog', '9']`.

<details><summary>Hint</summary>

Walk the string once with a buffer of alnum characters; flush it to the output whenever you hit a separator — and once more after the loop.

</details>


In [ ]:
import numpy as np

def tokenize(text):
    # your code — keep alnum runs, split on everything else
    return text.lower().split()  # wrong — leaves punctuation attached

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    words = [\"cat\", \"Dog\", \"BOBAI\", \"x9\", \"ml\", \"ai2\", \"sun\", \"ner\"]\n    seps = [\" \", \",\", \".\", \"!\", \"-\", \"  \"]\n    n = int(rng.integers(3, 10))\n    ws = [str(w) for w in rng.choice(words, n)]\n    parts = [w + str(rng.choice(seps)) for w in ws[:-1]] + [ws[-1]]\n    if rng.random() < 0.5:\n        parts[-1] += str(rng.choice(seps))\n    # ~half the texts end in a bare token — a tokenizer that forgets the final flush must fail\n    return {\"text\": \"\".join(parts)}\n\ndef tokenize(text):\n    toks, cur = [], []\n    for ch in text.lower():\n        if ch.isalnum():\n            cur.append(ch)\n        elif cur:\n            toks.append(\"\".join(cur)); cur = []\n    if cur:\n        toks.append(\"\".join(cur))\n    return toks", "tokenize", 20)

### Task 4 — Build the vocabulary dict

●○○ · ~6 min

> A vocabulary maps each distinct token to an integer id — the first step before any count vector or embedding lookup. Numbering in order of first appearance keeps ids stable and reproducible.

Implement `build_vocab(docs)` → dict mapping each distinct token to its index, numbered 0, 1, 2, … in order of first appearance across `docs` (a list of token lists).

<details><summary>Hint</summary>

One pass: `if t not in vocab: vocab[t] = len(vocab)`.

</details>


In [ ]:
import numpy as np

def build_vocab(docs):
    vocab = {}
    # your code — watch the ordering rule in the prompt
    return vocab

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    vocab = [\"alpha\", \"beta\", \"gamma\", \"delta\", \"eps\", \"zeta\"]\n    m = int(rng.integers(2, 6))\n    docs = [[str(w) for w in rng.choice(vocab, int(rng.integers(2, 7)))]\n            for _ in range(m)]\n    return {\"docs\": docs}\n\ndef build_vocab(docs):\n    vocab = {}\n    for d in docs:\n        for t in d:\n            if t not in vocab:\n                vocab[t] = len(vocab)\n    return vocab", "build_vocab", 20)

### Task 5 — Word n-grams

●●○ · ~8 min

> Word n-grams capture local context that bag-of-words loses ('not good' ≠ 'good'). They are just sliding windows over the token list — and the building block of every statistical language model.

Implement `ngrams(tokens, n)` → list of tuples of `n` consecutive tokens. `ngrams(['a','b','c'], 2)` → `[('a','b'), ('b','c')]`. If `n` exceeds the number of tokens, return an empty list.

<details><summary>Hint</summary>

[tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)] — the range is empty when the bound is non-positive.

</details>


In [ ]:
import numpy as np

def ngrams(tokens, n):
    # your code
    return []

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    toks = [str(w) for w in rng.choice([\"a\", \"b\", \"c\", \"d\", \"e\", \"f\"],\n                                      int(rng.integers(3, 12)))]\n    return {\"tokens\": toks, \"n\": int(rng.integers(1, 5))}\n\ndef ngrams(tokens, n):\n    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]", "ngrams", 20)

### Task 6 — TF-IDF matrix from scratch

●●○ · ~30 min

> TF-IDF turns documents into vectors where words frequent in this doc but rare overall get high weight: `tf(t,d) = count(t,d)/len(d)` and `idf(t) = ln(N/df(t)) + 1`, where `df` counts documents containing `t`. It is still the strongest zero-ML baseline for search and dedup tasks.

Implement `tfidf(docs)` → a (len(docs), vocab_size) float matrix over the alphabetically sorted vocabulary of `docs` (a list of token lists). Cell = tf·idf with `tf(t,d) = count(t,d)/len(d)` and `idf(t) = log(N/df(t)) + 1` (natural log, `df` = number of docs containing `t`).

<details><summary>Hint</summary>

vocab = sorted(union of tokens); accumulate df via set(d) per doc; fill the tf matrix, then multiply each column by its idf in one broadcasted op.

</details>


In [ ]:
import numpy as np

def tfidf(docs):
    vocab = sorted({t for d in docs for t in d})
    # your code
    return np.zeros((len(docs), len(vocab)))

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    words = [\"red\", \"blue\", \"green\", \"cat\", \"dog\", \"ml\", \"ai\"]\n    m = int(rng.integers(2, 7))\n    docs = [[str(w) for w in rng.choice(words, int(rng.integers(2, 8)))]\n            for _ in range(m)]\n    return {\"docs\": docs}\n\ndef tfidf(docs):\n    vocab = sorted({t for d in docs for t in d})\n    col = {t: i for i, t in enumerate(vocab)}\n    N = len(docs)\n    df = np.zeros(len(vocab))\n    for d in docs:\n        for t in set(d):\n            df[col[t]] += 1\n    idf = np.log(N / df) + 1.0\n    M = np.zeros((N, len(vocab)))\n    for i, d in enumerate(docs):\n        for t in d:\n            M[i, col[t]] += 1.0 / len(d)\n    return M * idf", "tfidf", 20)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/13-TextRep)